# 13.3 - Edges & Conditional Routing

**Phase:** 13 - LangGraph / Stateful Workflows

**Status:** VERIFIED

---

## 1. What Are We Solving?

Linear graphs cannot branch. `add_conditional_edges` lets the graph inspect state at runtime and choose which node to visit next — the graph's version of if/else and switch.

## 2. Why Does This Matter?

## 3. Prerequisites

Units 13.1-13.2 (manual state machine, LangGraph basics).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Write pure router functions that return a routing key
- Map routing keys to node names and wire `add_conditional_edges`
- Guarantee every branch path terminates at `END`
- Use routing for LLM intent classification

## 5. Mental Model

A conditional edge is a fork in the road: you reach an intersection (a node), consult a map (state), and take the road that leads to the right destination (node).


## 6. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: LangGraph is a stateful orchestration framework."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:  # network / quota / model errors -> never crash the notebook
        return f"[llm-error: {type(e).__name__}]"

print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


## 7. Implementation-First: Content Router

Pure rules first, no LLM — detect `code` / `text` / `data`, branch, and let every branch terminate.

In [2]:
class ContentState(TypedDict):
    content: str
    content_type: str | None
    result: str | None


def detect(state):
    c = state["content"]
    if "def " in c or "import " in c:
        state["content_type"] = "code"
    elif "," in c and "\n" in c:
        state["content_type"] = "data"
    else:
        state["content_type"] = "text"
    return state


def process_code(state):
    state["result"] = f"Code: {len(state['content'].splitlines())} lines"
    return state


def process_text(state):
    state["result"] = f"Text: {len(state['content'].split())} words"
    return state


def process_data(state):
    state["result"] = f"Data: {len(state['content'].strip().splitlines())} rows"
    return state


def router(state) -> str:
    return state["content_type"]


g = StateGraph(ContentState)
g.add_node("detect", detect)
g.add_node("code", process_code)
g.add_node("text", process_text)
g.add_node("data", process_data)
g.add_edge(START, "detect")
g.add_conditional_edges("detect", router, {"code": "code", "text": "text", "data": "data"})
g.add_edge("code", END)
g.add_edge("text", END)
g.add_edge("data", END)
app = g.compile()

for c in ["import os\nprint(os.getcwd())",
          "This is only a short sentence.",
          "a, b, c\n1, 2, 3\n4, 5, 6"]:
    r = app.invoke({"content": c, "content_type": None, "result": None})
    print(f"{r['content_type']:>6} -> {r['result']}")


  code -> Code: 2 lines
  text -> Text: 6 words
  data -> Data: 3 rows


## 8. LLM-Routed Support Assistant

Classify the user intent with Groq, then route to a specialized handler.

In [3]:
class IntentState(TypedDict):
    query: str
    intent: str | None
    answer: str | None


def classify_intent(state):
    prompt = ("Classify the question into exactly one intent: 'howto', 'debug', 'compare'. "
              "Reply with ONE word.\nQuestion: " + state["query"])
    tag = llm(prompt).strip().lower()
    tag = next((t for t in ("howto", "debug", "compare") if t in tag), "howto")
    state["intent"] = tag
    return state


def answer_howto(state):
    state["answer"] = "HOWTO> " + llm(f"Give step-by-step instructions for: {state['query']}")
    return state


def answer_debug(state):
    state["answer"] = "DEBUG> " + llm(f"Help debug this problem: {state['query']}")
    return state


def answer_compare(state):
    state["answer"] = "COMPARE> " + llm(f"Compare the options in: {state['query']}")
    return state


def intent_router(state):
    return state["intent"]


g2 = StateGraph(IntentState)
g2.add_node("classify", classify_intent)
g2.add_node("howto", answer_howto)
g2.add_node("debug", answer_debug)
g2.add_node("compare", answer_compare)
g2.add_edge(START, "classify")
g2.add_conditional_edges("classify", intent_router,
                         {"howto": "howto", "debug": "debug", "compare": "compare"})
for n in ("howto", "debug", "compare"):
    g2.add_edge(n, END)
app2 = g2.compile()

for q in ["How do I write a for loop?",
          "My model keeps overfitting, what should I do?",
          "Compare RAG to fine-tuning."]:
    r = app2.invoke({"query": q, "intent": None, "answer": None})
    print(f"{r['intent']:>7} -> {r['answer']}")


  howto -> HOWTO> Below is a **step‑by‑step guide** to writing a `for` loop.  
I’ll cover the core idea that applies to almost every language, then give a quick example in a few popular languages (Python, JavaScript, Java, C/C++).  

---

## 1. Understand the Purpose of a `for` Loop

A `for` loop is used when you know **how many times** you want to repeat a block of code.  
Typical structure:

```
for (initialization; condition; update) {
    // body – code that runs each iteration
}
```

- **Initialization** – set up a counter (or counters) before the loop starts.  
- **Condition** – a boolean expression that is checked before each iteration; if it’s `true`, the loop continues.  
- **Update** – executed after each iteration; usually increments or decrements the counter.  
- **Body** – the code that runs on every iteration.

---

## 2. Step‑by‑Step Instructions

| Step | What to Do | Why it Matters |
|------|------------|----------------|
| **1. Pick a counter variable** | Choose a nam

  howto -> HOWTO> ## How to Stop Your Model from Over‑Fitting  
*(Step‑by‑Step Guide)*  

> **TL;DR**  
> 1. **Diagnose** – confirm that training loss ↓ while validation loss ↑.  
> 2. **Simplify** – reduce model size or add regularization.  
> 3. **Regularize** – use dropout, weight decay, batch‑norm, etc.  
> 4. **Augment** – increase data diversity.  
> 5. **Early‑stop** – halt training when validation stops improving.  
> 6. **Validate** – use proper cross‑validation or a hold‑out set.  
> 7. **Iterate** – tweak one thing at a time, monitor metrics.

Below is a detailed, actionable workflow you can follow in any deep‑learning framework (PyTorch, TensorFlow, Keras, etc.).

---

### 1. Confirm the Problem

| What to check | Why it matters | How to check |
|---------------|----------------|--------------|
| **Training vs. Validation loss** | Over‑fitting shows training loss decreasing while validation loss increases. | Plot `train_loss` and `val_loss` per epoch. |
| **Training vs. Val

compare -> COMPARE> 


## 9. Merging Branches

After the specialized handler, you can route all branches to a common summary node instead of `END` — the graph merges naturally because nodes only need one incoming path.


## Common Mistakes

- **Return `None` from a node** — the graph silently drops the update. Always `return state`.
- **Mutating state in the router** — routers must be side-effect free.
- **Forgetting the terminal condition** — cycles run forever without an iteration guard.
- **Typo in a state key** — `TypedDict` catches it at compile time; plain dicts do not.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| `KeyError` on state | Field name mismatch | Match state keys to the `TypedDict` exactly |
| Graph won't compile | Node/referenced name typo | Check every string passed to `add_node`/`add_edge` |
| Node output ignored | Node returns `None` or a partial dict | Always `return state` (or a merge-able partial) |
| Infinite loop | No convergence guard | Add `max_steps` to state and check it in the router |
| Wrong branch taken | Router priority bug | Unit-test the router on every input variant |

## Best Practices

- Define all state fields upfront with defaults in a `TypedDict`.
- Keep node functions pure and focused: one responsibility each.
- Name nodes descriptively (`retrieve`, `generate`, not `step1`).
- Always add an iteration guard on loops.
- Inspect the graph with `app.get_graph().draw_mermaid()`.

## Hands-On Practice

1. **Basic:** Rerun the examples with new inputs; verify the trace.
2. **Guided:** Add a node that validates output before terminating.
3. **Independent:** Build a 3-step pipeline (fetch -> process -> summarize) with a retry node.
4. **Realistic:** Turn the example into a multi-department support agent.
5. **Challenge:** Save/load the state dict to JSON and resume the workflow from a checkpoint.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
